# 01 — LVVI3 data preprocessing

**Step 01 of the LVVI3 pipeline** (`scripts/LVVI3/data_preprocessing/`).

- **Purpose**: build the analysis-ready LVVI3 survey table from the raw survey extract.
  Activities are decoded from the one-shot `B1_*` / `B2_*` columns with the full
  questionnaire option mapping kept **in this notebook**; the same mapping is written to
  `data/LVVI3_activity_mapping.csv` as a middle product.
- **Inputs**: `data/YouTube/LVVI3_VIMAS_Leyi_Stateowned_land.xlsx` (raw survey data, sheet
  `LVVI3_VIMAS_Leyi_Stateowned_lan`; the copy under `data/LVVI3/` —
  `LVVI3_VIMAS_Leyi_Stateowned_land_variables.xlsx` — is labels-only and is used here for
  the variable/option labels), `data/LVVI3/LVVI3_activities.csv` (activity taxonomy:
  sub_activity → activity).
- **Outputs**: `data/LVVI3/lvvi3_or.csv`, `data/LVVI3/lvvi3_nbt.csv`,
  `data/LVVI3/lvvi3_or_en.csv`, `data/LVVI3/lvvi3_nbt_en.csv` (intermediates),
  `data/LVVI3/lvvi3_survey_en.csv` (main survey table, one row per respondent ×
  reported activity, with area-type flags `national_park` / `wilderness_area` /
  `nature_reserve` / `commercial_forest` harmonized with the social media layers
  — read by all RQ notebooks), `data/LVVI3_activity_mapping.csv` (mapping middle
  product).

**One-shot activity columns.** The questionnaire asks for the *main activity* of the last
trip: the activity group is coded in `B1_1` (day trip) / `B2_1` (multi-day trip), and the
detailed option is one-shot coded in the matching `B1_<activity>` / `B2_<activity>` column
(e.g. `B1_kavely` for walking). Respondents can report several activities when several of
these columns are filled; the melt-based extraction below keeps them all (one row per
respondent × activity). In the current extract each respondent has exactly one filled
activity column — see the validation section at the end.

## 1. Read raw data and variable labels

In [1]:
import numpy as np
import pandas as pd

RAW_XLSX = "../../../data/YouTube/LVVI3_VIMAS_Leyi_Stateowned_land.xlsx"   # holds the data sheet
LABEL_XLSX = "../../../data/LVVI3/LVVI3_VIMAS_Leyi_Stateowned_land_variables.xlsx"  # labels only

df_lvvi3 = pd.read_excel(RAW_XLSX, sheet_name="LVVI3_VIMAS_Leyi_Stateowned_lan")
print("raw survey:", df_lvvi3.shape, "| respondents:", df_lvvi3["kohdenro"].nunique())

raw survey: (636, 242) | respondents: 636


## 2. Preprocess data

In [2]:
# Replace placeholder values with NaN
df_lvvi3 = df_lvvi3.replace([99, 999], np.nan)

# Fill missing values in A51_km based on A51_m
df_lvvi3["A51_km"] = df_lvvi3.apply(
    lambda row: row["A51_m"] / 1000 if pd.isna(row["A51_km"]) and pd.notna(row["A51_m"]) else row["A51_km"],
    axis=1)

# Remove rows with missing values in following columns
df_lvvi3 = df_lvvi3.dropna(subset=["A50", "A51_km", "A52", "A_kansallispuisto", "A_retkeilyalue", "TA49h"])
print("after dropna:", df_lvvi3.shape)

after dropna: (557, 242)


In [3]:
# Activity data preprocessing
# Remove rows with missing values in B1_1 and B2_1
df_lvvi3 = df_lvvi3[~((df_lvvi3["B1_1"].isnull()) & (df_lvvi3["B2_1"].isnull()))]

# Remove rows with value 23 in B1_1 and B2_1 (inconsistent coding)
df_lvvi3 = df_lvvi3[df_lvvi3["B1_1"] != 23]
df_lvvi3 = df_lvvi3[df_lvvi3["B2_1"] != 23]

# Remove respondent with kohdenro 10202 as his/her activity data is inconsistent
# (B1_1 = walking but only the running column is filled)
df_lvvi3 = df_lvvi3[df_lvvi3["kohdenro"] != 10202]
print("after activity cleaning:", df_lvvi3.shape)

after activity cleaning: (484, 242)


## 3. Activity decoding mapping (kept in this notebook)

Each of the 21 activity columns per trip section holds the code of the detailed activity
option. The mapping is defined here in three layers:

1. **questionnaire options** — code → option text, from the `Labels_en` sheet of the
   variables workbook (shown below so the notebook is self-contained);
2. **canonicalization** — option code → normalized `sub_activity` (`SUB_ACTIVITY_CANONICAL`
   below; this replaces the earlier ad-hoc `str.replace` chains);
3. **taxonomy** — `sub_activity` → `activity` via `LVVI3_activities.csv`, and
   `activity` → `structural_activity` via `STRUCTURAL_BY_ACTIVITY` (assignments follow the
   combinations observed in the previous survey table; activities unobserved in this data
   are assigned by analogy).

The combined mapping is written to `data/LVVI3_activity_mapping.csv` as a middle product.

In [4]:
# 3a. Questionnaire options per one-shot column, parsed from the labels workbook
import re

labels_en = pd.read_excel(LABEL_XLSX, sheet_name="Labels_en")
labels_en.columns = ["variable", "label", "values"]
OPTION_PATTERN = re.compile(r"\{(\d+),\s*([^}]*)\}")

# (B1_1 activity category, Finnish column stem); category 19 (rollerblading) has no column
ACTIVITY_COLUMNS = [
    (1, "kavely"), (2, "pyora"), (3, "juoksu"), (4, "koira"), (5, "lapsi"),
    (6, "teltta"), (7, "vene"), (8, "uinti"), (9, "retki"), (10, "metsa"),
    (11, "kasvi"), (12, "kala"), (13, "hirvi"), (14, "marja"), (15, "kuvaus"),
    (16, "hiihto"), (17, "luistelu"), (18, "aarre"), (20, "golf"), (21, "mopo"), (22, "opas"),
]
STEM_TO_CAT = {stem: cat for cat, stem in ACTIVITY_COLUMNS}

def parse_options(variable):
    row = labels_en.loc[labels_en["variable"] == variable, "values"]
    if len(row) == 0:
        return {}
    return {int(m.group(1)): m.group(2).strip() for m in OPTION_PATTERN.finditer(str(row.values[0]))}

# option text per (section, column stem, code) — the detailed original options
OPTION_LABELS = {
    (section, stem, code): option
    for section in ("B1", "B2")
    for _, stem in ACTIVITY_COLUMNS
    for code, option in parse_options(f"{section}_{stem}").items()
}
print(f"{len(OPTION_LABELS)} questionnaire options decoded")

172 questionnaire options decoded


In [5]:
# 3b. Canonicalization: (section, column stem, option code) -> canonical sub_activity
SUB_ACTIVITY_CANONICAL = {
    ("B1", "kavely", 1): "Hiking", ("B1", "kavely", 2): "Hiking",
    ("B1", "kavely", 3): "Nordic walking", ("B1", "kavely", 4): "Walking",
    ("B2", "kavely", 1): "Hiking", ("B2", "kavely", 2): "Hiking",
    ("B2", "kavely", 3): "Nordic walking", ("B2", "kavely", 4): "Walking",
    ("B1", "pyora", 1): "Mountain biking", ("B1", "pyora", 2): "Cycling", ("B1", "pyora", 3): "Cycling",
    ("B2", "pyora", 1): "Mountain biking", ("B2", "pyora", 2): "Cycling",
    ("B2", "pyora", 3): "Cycling", ("B2", "pyora", 4): "Cycling",
    ("B1", "juoksu", 1): "Jogging", ("B1", "juoksu", 2): "Running",
    ("B2", "juoksu", 1): "Jogging", ("B2", "juoksu", 2): "Running",
    ("B1", "koira", 1): "Dog walking", ("B1", "koira", 2): "Dog sledding in nature",
    ("B2", "koira", 1): "Dog walking", ("B2", "koira", 2): "Dog sledding in nature",
    ("B1", "lapsi", 1): "Outdoor activities with children", ("B1", "lapsi", 2): "Games and play in nature",
    ("B2", "lapsi", 1): "Outdoor activities with children", ("B2", "lapsi", 2): "Games and play in nature",
    ("B1", "teltta", 1): "Staying in a summer cottage, farm, holiday home",
    ("B1", "teltta", 2): "Travelling or staying in a caravan or camper",
    ("B1", "teltta", 3): "Camping",
    ("B1", "teltta", 4): "Staying overnight in a tent, lodge, etc. in natural terrain",
    ("B2", "teltta", 1): "Staying in a summer cottage, farm, holiday home",
    ("B2", "teltta", 2): "Travelling or staying in a caravan or camper",
    ("B2", "teltta", 3): "Camping",
    ("B2", "teltta", 4): "Staying overnight in a tent, lodge, etc. in natural terrain",
    ("B1", "vene", 1): "Rowing boat", ("B1", "vene", 2): "Motorboat cruising",
    ("B1", "vene", 3): "Motorboat small boat", ("B1", "vene", 4): "Sailing cruising",
    ("B1", "vene", 5): "Sailing small boat", ("B1", "vene", 6): "Canoeing",
    ("B1", "vene", 7): "Surfing", ("B1", "vene", 8): "SUP boating",
    ("B2", "vene", 1): "Rowing boat", ("B2", "vene", 2): "Motorboat cruising",
    ("B2", "vene", 3): "Motorboat small boat", ("B2", "vene", 4): "Sailing cruising",
    ("B2", "vene", 5): "Sailing small boat", ("B2", "vene", 6): "Canoeing",
    ("B2", "vene", 7): "Surfing", ("B2", "vene", 8): "SUP boating", ("B2", "vene", 9): "Water sports",
    ("B1", "uinti", 1): "Swimming", ("B1", "uinti", 2): "Swimming",
    ("B1", "uinti", 3): "Swimming", ("B1", "uinti", 4): "Diving",
    ("B2", "uinti", 1): "Swimming", ("B2", "uinti", 2): "Swimming",
    ("B2", "uinti", 3): "Swimming", ("B2", "uinti", 4): "Diving",
    ("B1", "retki", 1): "Picnicking", ("B1", "retki", 2): "Beach games",
    ("B1", "retki", 3): "Sunbathing", ("B1", "retki", 4): "Sunbathing",
    ("B2", "retki", 1): "Picnicking", ("B2", "retki", 2): "Beach games",
    ("B2", "retki", 3): "Sunbathing", ("B2", "retki", 4): "Sunbathing",
    ("B1", "metsa", 1): "Fuelwood procurement, collection of twigs, branches and trees for domestic use",
    ("B1", "metsa", 2): "Forestry work such as planting, nursery management, etc.",
    ("B1", "metsa", 3): "Game management",
    ("B1", "metsa", 4): "Fish water restoration",
    ("B1", "metsa", 5): "Management of meadows or other natural habitats",
    ("B2", "metsa", 1): "Fuelwood procurement, collection of twigs, branches and trees for domestic use",
    ("B2", "metsa", 2): "Forestry work such as planting, nursery management, etc.",
    ("B2", "metsa", 3): "Game management",
    ("B2", "metsa", 4): "Fish water restoration",
    ("B2", "metsa", 5): "Management of meadows or other natural habitats",
    ("B1", "kasvi", 1): "Growing ornamental plants, vegetables or cereals in leisure time",
    ("B1", "kasvi", 2): "Keeping livestock in a natural environment in leisure time",
    ("B2", "kasvi", 1): "Growing ornamental plants, vegetables or cereals in leisure time",
    ("B2", "kasvi", 2): "Keeping livestock in a natural environment in leisure time",
    ("B1", "kala", 1): "Worm fishing", ("B1", "kala", 2): "Trolling", ("B1", "kala", 3): "Spotting",
    ("B1", "kala", 4): "Fly fishing",
    ("B1", "kala", 5): "Fishing with net, line, catfish, trap etc.",
    ("B1", "kala", 6): "Fishing with long line, hook etc.", ("B1", "kala", 7): "Crabbing",
    ("B2", "kala", 1): "Worm fishing", ("B2", "kala", 2): "Trolling", ("B2", "kala", 3): "Spotting",
    ("B2", "kala", 4): "Fly fishing",
    ("B2", "kala", 5): "Fishing with net, line, catfish, trap etc.",
    ("B2", "kala", 6): "Fishing with long line, hook etc.", ("B2", "kala", 7): "Crabbing",
    ("B1", "hirvi", 1): "Deer hunting", ("B1", "hirvi", 2): "Bird hunting",
    ("B2", "hirvi", 1): "Deer hunting", ("B2", "hirvi", 2): "Bird hunting",
    ("B1", "marja", 1): "Berry picking", ("B1", "marja", 2): "Mushroom picking",
    ("B1", "marja", 3): "Picking herbs, flowers or ornamental plants in nature",
    ("B2", "marja", 1): "Berry picking", ("B2", "marja", 2): "Mushroom picking",
    ("B2", "marja", 3): "Picking herbs, flowers or ornamental plants in nature",
    ("B1", "kuvaus", 1): "Nature photography", ("B1", "kuvaus", 2): "Bird watching",
    ("B1", "kuvaus", 3): "Nature observation", ("B1", "kuvaus", 4): "Other nature observation",
    ("B2", "kuvaus", 1): "Nature photography", ("B2", "kuvaus", 2): "Bird watching",
    ("B2", "kuvaus", 3): "Nature observation", ("B2", "kuvaus", 4): "Other nature observation",
    ("B1", "hiihto", 1): "Cross-country skiing", ("B1", "hiihto", 2): "Cross-country skiing",
    ("B1", "hiihto", 3): "Cross-country skiing", ("B1", "hiihto", 4): "Cross-country skiing",
    ("B1", "hiihto", 5): "Downhill skiing", ("B1", "hiihto", 6): "Snowboarding",
    ("B1", "hiihto", 7): "Off-piste skiing or snowboarding off slopes",
    ("B2", "hiihto", 1): "Cross-country skiing", ("B2", "hiihto", 2): "Cross-country skiing",
    ("B2", "hiihto", 3): "Cross-country skiing", ("B2", "hiihto", 4): "Cross-country skiing",
    ("B2", "hiihto", 5): "Downhill skiing", ("B2", "hiihto", 6): "Snowboarding",
    ("B2", "hiihto", 7): "Off-piste skiing or snowboarding off slopes",
    ("B1", "luistelu", 1): "Skating", ("B1", "luistelu", 2): "Skating",
    ("B1", "luistelu", 3): "Kicksledding", ("B1", "luistelu", 4): "Snowshoeing",
    ("B1", "luistelu", 5): "Sledding",
    ("B2", "luistelu", 1): "Skating", ("B2", "luistelu", 2): "Skating",
    ("B2", "luistelu", 3): "Snowmobiling", ("B2", "luistelu", 4): "Snowshoeing",
    ("B2", "luistelu", 5): "Sledding",
    ("B1", "aarre", 1): "Orienteering", ("B1", "aarre", 2): "Treasure hunting",
    ("B1", "aarre", 3): "Rock or mountain climbing", ("B1", "aarre", 4): "Horseback riding",
    ("B2", "aarre", 1): "Orienteering", ("B2", "aarre", 2): "Treasure hunting",
    ("B2", "aarre", 3): "Rock or mountain climbing", ("B2", "aarre", 4): "Cross-country skiing",
    ("B1", "golf", 1): "Golf", ("B1", "golf", 2): "Flying a drone", ("B1", "golf", 3): "Frisbee golf",
    ("B2", "golf", 1): "Golf", ("B2", "golf", 2): "Flying a drone", ("B2", "golf", 3): "Frisbee golf",
    ("B1", "mopo", 1): "Snowmobiling", ("B1", "mopo", 2): "ATV riding",
    ("B1", "mopo", 3): "Four-wheel-drive off-road driving", ("B1", "mopo", 4): "Off-road motorcycling",
    ("B1", "mopo", 5): "Other motorised outdoor activities",
    ("B2", "mopo", 1): "Snowmobiling", ("B2", "mopo", 2): "ATV riding",
    ("B2", "mopo", 3): "Four-wheel-drive off-road driving", ("B2", "mopo", 4): "Off-road motorcycling",
    ("B2", "mopo", 5): "Other motorised outdoor activities",
    ("B1", "opas", 1): "Free guided excursion", ("B1", "opas", 2): "Guided excursion for a fee",
    ("B1", "opas", 3): "Scout excursion",
    ("B1", "opas", 4): "Group exercise e.g. gymnastics or dance in a park or in nature",
    ("B1", "opas", 5): "Outdoor event or competition in nature",
    ("B2", "opas", 1): "Free guided excursion", ("B2", "opas", 2): "Guided excursion for a fee",
    ("B2", "opas", 3): "Scout excursion",
    ("B2", "opas", 4): "Group exercise e.g. gymnastics or dance in a park or in nature",
    ("B2", "opas", 5): "Outdoor event or competition in nature",
}

**Notes on specific assignments**

- `B2_pyora` codes 3 and 4 are duplicate options in the questionnaire ("other cycling or
  cycling for leisure") and both map to `Cycling`.
- `B2_luistelu` code 3 is translated as "recreational snowmobiling" in the questionnaire
  (likely a translation glitch in the skating column); it is mapped literally to
  `Snowmobiling`.
- `B2_aarre` code 4 is translated as "cross-country skiing" (B1 has "horseback riding in
  natural terrain" here); mapped literally to `Cross-country skiing`.
- `golf` code 2 ("flying a kite, drone or drone") maps to `Flying a drone`.
- `retki` is the storage column for activity category 9 (picnics / beach / sunbathing);
  the column name suggests hiking, but its options are the picnic options.

In [6]:
# 3c. Taxonomy: canonical sub_activity -> activity (LVVI3_activities.csv) and
#     activity -> structural_activity
taxonomy = pd.read_csv("../../../data/LVVI3/LVVI3_activities.csv").iloc[:, :2]
sub_to_activity = (
    taxonomy.drop_duplicates("sub_activity")
    .set_index("sub_activity")["activity"].to_dict()
)
sub_to_activity.update({
    "Dog sledding in nature": "Outdoor activities with dogs",  # not in the survey taxonomy
})

STRUCTURAL_BY_ACTIVITY = {
    "Walking": "Leisure", "Running": "Leisure", "Cycling": "Adventure",
    "Skiing": "Adventure", "Skating": "Adventure", "Snowmobiling": "Adventure",
    "Swimming": "Adventure", "Diving": "Adventure", "Climbing": "Adventure",
    "Orienteering": "Adventure", "Rollerblading": "Adventure", "Sledding": "Adventure",
    "Horseback riding": "Adventure", "Treasure hunting": "Adventure",
    "Boating": "Leisure", "Dog walking": "Leisure",
    "Outdoor activities with dogs": "Leisure",
    "Outdoor activities with children": "Leisure", "Picnicking": "Leisure",
    "Frisbee golf": "Leisure", "Golf": "Leisure",
    "Flying a kite": "Leisure", "Flying a drone": "Leisure",
    "Fishing": "Consumptive interaction", "Hunting": "Consumptive interaction",
    "Nature product collection": "Consumptive interaction",
    "Bird watching": "Nature Appreciation", "Nature observation": "Nature Appreciation",
    "Nature photography": "Nature Appreciation",
    "Camping": "Nature Immersion",
    "Travelling or staying in a caravan or camper": "Nature Immersion",
    "Staying in a summer cottage, farm, holiday home": "Nature Immersion",
    "Staying overnight in a tent, lodge, etc. in natural terrain": "Nature Immersion",
    "Guided tour": "Organized Experience",
    "Nature management": "Land Management", "Gardening": "Land Management",
}

### 3d. Multi-activity extraction

Melt the one-shot activity columns of a trip section into long format: one row per
respondent × filled activity column.

In [7]:
def extract_activities(df_sec, section):
    """Melt the one-shot activity columns: one row per respondent x filled activity column."""
    cols = [f"{section}_{stem}" for _, stem in ACTIVITY_COLUMNS if f"{section}_{stem}" in df_sec.columns]
    long = (
        df_sec[["kohdenro"] + cols]
        .melt(id_vars="kohdenro", var_name="survey_column", value_name="sub_activity_cat")
        .dropna(subset=["sub_activity_cat"])
        .assign(sub_activity_cat=lambda d: d["sub_activity_cat"].astype(int))
    )
    long["column_stem"] = long["survey_column"].str.replace(f"{section}_", "", n=1)
    long["activity_cat"] = long["column_stem"].map(STEM_TO_CAT)
    long["sub_activity_option"] = long.apply(
        lambda r: OPTION_LABELS.get((section, r["column_stem"], r["sub_activity_cat"])), axis=1)
    long["sub_activity"] = long.apply(
        lambda r: SUB_ACTIVITY_CANONICAL.get((section, r["column_stem"], r["sub_activity_cat"])), axis=1)
    long["activity"] = long["sub_activity"].map(sub_to_activity)
    long["structural_activity"] = long["activity"].map(STRUCTURAL_BY_ACTIVITY)
    return long

## 4. Distinguish outdoor recreation and nature-based tourism participants

The same respondents split into two trip sections: `B1_*` (day trip, outdoor recreation)
and `B2_*` (multi-day trip, nature-based tourism).

In [8]:
# Column groups kept for each section (common columns + section-specific columns)
common_columns = ["kohdenro", "paino2", "A1", "A2", "A48", "TA49h", "A50", "A51_km",
                  "A_paiva1", "A_paiva2", "A_paiva3a", "A_paiva3b", "A52", "A53a", "A53b",
                  "A53c", "A53d", "A53e", "A53f", "A53muu", "A54", "A55",
                  "A_kansallispuisto", "A_retkeilyalue", "A_kansallispuisto_krt",
                  "A_retkeilyalue_krt", "Ymp1a", "Ymp1b", "Ymp1c", "Ymp1d", "Ymp1e",
                  "Ymp1f", "Ymp1g", "Ymp1h", "Ymp1i", "Ymp1j", "Ymp1l", "Ymp1m",
                  "Ymp2a", "Ymp2b", "Ymp2c", "Ymp2d", "Ymp2e", "Ymp2f", "Ymp2g", "Ymp2h",
                  "Ymp2i", "Ymp3a", "Ymp3b", "Ymp3c", "Ymp3d", "Ymp4a", "Ymp4b", "Ymp4c",
                  "Ymp4d", "Ymp4e", "Ymp4f", "A_GPSkello", "motivation_1", "motivation_2",
                  "motivation_3", "motivation_4", "motivation_5", "motivation_6",
                  "motivation_7", "motivation_8", "motivation_9", "motivation_10",
                  "area_destination", "area_municipality", "area_classification",
                  "area_name", "B_2", "B_2a", "B_2b", "B_6a", "B_6b", "B_6c", "B_6d",
                  "B_6e", "B_6f", "B_6g", "B_6h", "B_6i", "B_6j", "B_6k", "B_6l", "B_6m",
                  "B_6n", "B_6muu", "B_7a", "B_7b", "B_7suo", "B_7c", "B_7d", "B_7e",
                  "B_7f", "B_7g", "B_7h", "B_7i", "B_7j", "B_7k", "B_7l", "B_7m", "B_7n",
                  "B_7muu", "B_typicality"]
or_columns = ["B1_3", "B1_3muu", "B1_8", "B1_8muu", "B1_9"]
nbt_columns = ["B2_6", "B2_8", "B2_11", "B2_12", "B2_seu", "A55kesto", "B2mataik",
               "B2saav", "B2_palv"]

### 4.1 Outdoor recreation participants (day trip, `B1_*`)

In [9]:
df_or = df_lvvi3[df_lvvi3["B1_1"].notnull()].copy()
df_or = df_or.merge(
    extract_activities(df_or, "B1")[["kohdenro", "sub_activity"]], on="kohdenro", how="left")

dict_or_hobbys = {1: "Walking", 2: "Running", 3: "Outdoor activities with dogs",
                  4: "Outdoor activities with children", 5: "Cycling",
                  6: "Cross-country skiing", 7: "Nature product collection",
                  8: "Nordic walking", 9: "Other activities"}
df_or["hobby"] = df_or["B1pharr"].map(dict_or_hobbys)

# Drop rows where any B1 motivation column is NA
B1_motivation_columns = [f"B1_motivaatio{i}" for i in range(1, 11)]
df_or = df_or[~df_or[B1_motivation_columns].isnull().any(axis=1)]
df_or = df_or.rename(columns={f"B1_motivaatio{i}": f"motivation_{i}" for i in range(1, 11)})

In [10]:
# Rename columns for uniforming columns in df_or and df_nbt
df_or = df_or.rename(columns={"B1_2": "B_2", "B1_2a": "B_2a", "B1_2b": "B_2b",
                              "lvkayt2": "area_classification", "B1_5a": "area_name",
                              **{f"B1_6{c}": f"B_6{c}" for c in list("abcdefghijklmn") + ["muu"]},
                              **{f"B1_7{c}": f"B_7{c}" for c in ["a", "b", "suo", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", "n", "muu"]},
                              "B1_tyypillisyys": "B_typicality"})

# Combined destination / municipality columns
df_or["area_destination"] = df_or.apply(
    lambda row: row["B1_4G"] if pd.isna(row["B1_4"]) and pd.notna(row["B1_4G"]) else row["B1_4"], axis=1)
df_or["area_municipality"] = df_or.apply(
    lambda row: row["B1_5c"] if pd.isna(row["B1K_2b"]) and pd.notna(row["B1_5c"]) else row["B1K_2b"], axis=1)
df_or["area_municipality"] = df_or["area_municipality"].apply(
    lambda x: x.lower().title() if pd.notna(x) else x)

# Remove rows with missing values in B_2
df_or = df_or.dropna(subset=["B_2"])

# Keep only the common columns, the decoded activity and outdoor recreation columns
df_or = df_or[common_columns[:21] + ["hobby"] + common_columns[21:]
              + ["sub_activity"] + or_columns]
print("or:", df_or.shape)

or: (165, 114)


### 4.2 Nature-based tourism participants (multi-day trip, `B2_*`)

In [11]:
df_nbt = df_lvvi3[df_lvvi3["B2_1"].notnull()].copy()
df_nbt = df_nbt.merge(
    extract_activities(df_nbt, "B2")[["kohdenro", "sub_activity"]], on="kohdenro", how="left")

dict_nbt_hobbys = {1: "Walking", 2: "Hiking", 3: "Camping", 4: "Cross-country skiing",
                   5: "Outdoor activities with dogs", 6: "Fishing", 7: "Skiing",
                   8: "Nature photography", 9: "Outdoor activities with children",
                   10: "Boating", 11: "Nature product collection", 12: "Other activities"}
df_nbt["hobby"] = df_nbt["B2pharr"].map(dict_nbt_hobbys)

# Drop rows where any B2 motivation column is NA
B2_motivation_columns = [f"B2_motivaatio{i}" for i in range(1, 11)]
df_nbt = df_nbt[~df_nbt[B2_motivation_columns].isnull().any(axis=1)]
df_nbt = df_nbt.rename(columns={f"B2_motivaatio{i}": f"motivation_{i}" for i in range(1, 11)})

In [12]:
# Fill missing B2_9 / B2_10 values from the B2K_4* follow-up columns
for c in "abcd":
    df_nbt[f"B2_9{c}"] = df_nbt.apply(
        lambda row: row[f"B2K_4{c}"] if pd.isna(row[f"B2_9{c}"]) else row[f"B2_9{c}"], axis=1)
for i, c in enumerate("ijklmn"):
    df_nbt[f"B2_10{c}"] = df_nbt.apply(
        lambda row: row[f"B2K_4{chr(ord('e') + i)}"] if pd.isna(row[f"B2_10{c}"]) else row[f"B2_10{c}"], axis=1)

# Map B2 columns to unified codes
df_nbt = df_nbt.rename(columns={"B2_2": "B_2", "B2_3": "B_2a", "B2_4": "B_2b",
                                "makayt2": "area_classification", "B2_7a": "area_name",
                                **{f"B2_9{c}": f"B_6{c}" for c in list("abcdefgh") + ["i", "j", "k", "l", "m", "n", "muu"]},
                                **{f"B2_10{c}": f"B_7{c}" for c in ["a", "b", "suo", "c", "d", "e", "f", "g", "h", "i", "j", "k", "l", "m", "n", "muu"]},
                                "B2_tyypillisyys": "B_typicality"})

# Combined destination / municipality columns
df_nbt["area_destination"] = df_nbt.apply(
    lambda row: row["B2K_3"] if pd.isna(row["B2_6"]) and pd.notna(row["B2K_3"]) else row["B2_6"], axis=1)
df_nbt["area_municipality"] = df_nbt.apply(
    lambda row: row["B2_7b"] if pd.isna(row["B2K_2b"]) and pd.notna(row["B2_7b"]) else row["B2K_2b"], axis=1)
df_nbt["area_municipality"] = df_nbt["area_municipality"].apply(
    lambda x: x.lower().title() if pd.notna(x) else x)

# Remove rows with missing values in B_2
df_nbt = df_nbt.dropna(subset=["B_2"])

# Keep only the common columns, the decoded activity and nature-based tourism columns
df_nbt = df_nbt[common_columns[:21] + ["hobby"] + common_columns[21:]
                + ["sub_activity"] + nbt_columns]
print("nbt:", df_nbt.shape)

nbt: (286, 118)


### 4.3 Correct municipality names

In [13]:
MUNICIPALITY_FIXES = {
    "Jällivaara, Ruotsi": None, "Ruovedellä": "Ruovesi", "Vasa": "Vaasa",
    "Kiiminki": "Oulu", "Äkäslompolo": "Kolari", "Ylläs": "Kolari",
    "Saariselkä": "Inari", "Raseborg": "Raasepori", "Evo, Asikkala": "Asikkala",
    "Helsingfors": "Helsinki", "Vantaa Ja Helsinki": "Vantaa", "Korpilahti": "Jyväskylä",
    "Uimaharju": "Joensuu", "Gran Canaria": None, "Hirvas": "Rovaniemi", "Åbo": "Turku",
    "Nykarleby": "Uusikaarlepyy", "Jakobstad": "Pedersören kunta",
    "Lammi, Hämeenlinna": "Hämeenlinna", "Kuru": "Ylöjärvi", "Hyrysalmi": "Hyrynsalmi",
    "Koskitl": "Koski Tl", "Kokkoa": "Kokkola", "Teijo": "Salo", "Vuokatti": "Sotkamo",
    "Levi": "Kittilä", "Ivalo ?": "Inari", "Pudasjärvi / Norra Österbotten": "Pudasjärvi",
    "Ivalo": "Inari", "Kilpisjärvi": "Enontekiö", "Lappi": "Rauma",
    "Varkaus-Kangaslampi": "Varkaus", "Raseborg (Tenala)": "Raasepori",
    "Inari, Lappi": "Inari", "Tulppio": "Savukoski", "Tuuri": "Alavus",
    "Näätämö": "Inari", "Kauhajoki?": "Kauhajoki", "Inari Ja Utsjoki": "Inari",
    "Nauvo": "Parainen", "Eri Kuntien Alueella": None, "Pedersöre": "Pedersören kunta",
    "Lapin Lääni": None, "Juankoski": "Kuopio", "Karunki": "Tornio", "Koli": "Lieksa",
    "Inari, Sodankylä": "Inari", "Nuuksio": "Espoo", "Mathildedal": "Salo",
    "Ruka Kuusamo": "Kuusamo", "Grönlanti": None, "Vahto-Säkylä Akseli": "Säkylä",
    "Ilomantsi Ja Inari": "Ilomantsi", "Pelkosenniemi Ja Savukoski": "Pelkosenniemi",
    "Lammi": "Hämeenlinna", "Kivijärvi/Perho/Kinnula": "Kivijärvi",
    "Haapajäri": "Haapajärvi", "Pohjois-Karjala": "Joensuu",
}
df_or["area_municipality"] = df_or["area_municipality"].replace(MUNICIPALITY_FIXES)
df_nbt["area_municipality"] = df_nbt["area_municipality"].replace(MUNICIPALITY_FIXES)

### 4.4 Translate column names into English and write the intermediates

In [14]:
RENAME_COMMON = {
    "kohdenro": "kohdenro_reference_num", "paino2": "paino2_weight",
    "A1": "A1_gender", "A2": "A2_age", "A48": "A48_last_activity_time",
    "TA49h": "TA49h_last_activity_duration", "A50": "A50_travel_time",
    "A51_km": "A51_km_travel_distance",
    "A_paiva1": "A_paiva1_day_trip_last_year", "A_paiva2": "A_paiva2_day_trip_last_year_times",
    "A_paiva3a": "A_paiva3a_day_trip_area", "A_paiva3b": "A_paiva3b_day_trip_city",
    "A52": "A52_multi_day_trip_last_year",
    "A53a": "A53a_multi_day_trip_last_year_cottage",
    "A53b": "A53b_multi_day_trip_last_year_lapland_winter",
    "A53c": "A53c_multi_day_trip_last_year_lapland_summer",
    "A53d": "A53d_multi_day_trip_last_year_archipelago",
    "A53e": "A53e_multi_day_trip_last_year_lake",
    "A53f": "A53f_multi_day_trip_last_year_other",
    "A53muu": "A53muu_multi_day_trip_last_year_other_specify",
    "A54": "A54_last_multi_day_trip_time", "A55": "A55_last_multi_day_trip_duration",
    "A_kansallispuisto": "A_national_park_last_year",
    "A_retkeilyalue": "A_recreational_area_last_year",
    "A_kansallispuisto_krt": "A_national_park_last_year_times",
    "A_retkeilyalue_krt": "A_recreational_area_last_year_times",
    "Ymp1a": "Ymp1a_concern_logging", "Ymp1b": "Ymp1b_concern_climate_change",
    "Ymp1c": "Ymp1c_concern_noise", "Ymp1d": "Ymp1d_concern_water_quality",
    "Ymp1e": "Ymp1e_concern_forest_reduction", "Ymp1f": "Ymp1f_concern_species_loss",
    "Ymp1g": "Ymp1g_concern_alien_species", "Ymp1h": "Ymp1h_concern_vegetarian",
    "Ymp1i": "Ymp1i_concern_oil_spill", "Ymp1j": "Ymp1j_concern_landscape_closure",
    "Ymp1l": "Ymp1l_concern_wind_farms", "Ymp1m": "Ymp1m_concern_harmful_substances",
    "Ymp2a": "Ymp2a_frequency_save_energy",
    "Ymp2b": "Ymp2b_frequency_energy_saving_products",
    "Ymp2c": "Ymp2c_frequency_household_waste", "Ymp2d": "Ymp2d_frequency_eco-products",
    "Ymp2e": "Ymp2e_frequency_organic_food", "Ymp2f": "Ymp2f_frequency_avoid_driving",
    "Ymp2g": "Ymp2g_frequency_avoid_flying", "Ymp2h": "Ymp2h_frequency_vegetarian",
    "Ymp2i": "Ymp2i_frequency_recycled_products",
    "Ymp3a": "Ymp3a_attitude_avoid_driving",
    "Ymp3b": "Ymp3b_attitude_more_travelling_time",
    "Ymp3c": "Ymp3c_attitude_nearby_destination",
    "Ymp3d": "Ymp3d_attitude_reduce_purchase_equipments",
    "Ymp4a": "Ymp4a_attitude_low_carbon_travel",
    "Ymp4b": "Ymp4b_attitude_more_travelling_time",
    "Ymp4c": "Ymp4c_attitude_nearby_destination",
    "Ymp4d": "Ymp4d_attitude_group_travel",
    "Ymp4e": "Ymp4e_attitude_low_carbon_services",
    "Ymp4f": "Ymp4f_attitude_avoid_trips",
    "A_GPSkello": "A_GPSkello_watch_use",
    "motivation_1": "motivation_1_peace", "motivation_2": "motivation_2_natural_products",
    "motivation_3": "motivation_3_landscapes", "motivation_4": "motivation_4_experiences",
    "motivation_5": "motivation_5_fitness", "motivation_6": "motivation_6_recovery",
    "motivation_7": "motivation_7_time", "motivation_8": "motivation_8_social",
    "motivation_9": "motivation_9_personal_skills", "motivation_10": "motivation_10_adventure",
    "B_2": "B_2_accompany", "B_2a": "B_2a_accompany_num", "B_2b": "B_2b_accompany_children",
    "B_6a": "B_6a_area_footpath", "B_6b": "B_6b_area_built_path",
    "B_6c": "B_6c_area_references", "B_6d": "B_6d_area_signposting",
    "B_6e": "B_6e_area_visitor_center", "B_6f": "B_6f_area_outdoor_house",
    "B_6g": "B_6g_area_deserted_house", "B_6h": "B_6h_area_fireplace",
    "B_6i": "B_6i_area_beach", "B_6j": "B_6j_area_camping_area",
    "B_6k": "B_6k_area_ski_trail", "B_6l": "B_6l_area_ski_slope",
    "B_6m": "B_6m_area_boat_dock", "B_6n": "B_6n_area_other",
    "B_6muu": "B_6muu_area_other_specify",
    "B_7a": "B_7a_lawn", "B_7b": "B_7b_forest", "B_7suo": "B_7suo_swamp",
    "B_7c": "B_7c_cultivated_field", "B_7d": "B_7d_meadow", "B_7e": "B_7e_mountain",
    "B_7f": "B_7f_lake", "B_7g": "B_7g_sea", "B_7h": "B_7h_river",
    "B_7i": "B_7i_berry", "B_7j": "B_7j_mushroom", "B_7k": "B_7k_hunting",
    "B_7l": "B_7l_fishing", "B_7m": "B_7m_natural_attraction", "B_7n": "B_7n_other",
    "B_7muu": "B_7muu_other_specify", "B_typicality": "B_typicality_usual_area",
}
RENAME_OR = {"B1_3": "B1_3_departure_destination",
             "B1_3muu": "B1_3muu_departure_destination_specify",
             "B1_8": "B1_8_transportation_mode", "B1_8muu": "B1_8muu_other_specify",
             "B1_9": "B1_9_same_destination_times"}
RENAME_NBT = {"B2_8": "B2_8_urban_rural_trip", "B2_11": "B2_11_travel_distance",
              "B2_12": "B2_12_travel_time", "B2_seu": "B2_accompany",
              "A55kesto": "A55kesto_trip_duration", "B2mataik": "B2mataik_travel_time",
              "B2saav": "B2saav_accessibility", "B2_palv": "B2_palv_services"}

df_or_en = df_or.rename(columns={**RENAME_COMMON, **RENAME_OR})
df_nbt_en = df_nbt.rename(columns={**RENAME_COMMON, **RENAME_NBT})

df_or_en.to_csv("../../../data/LVVI3/lvvi3_or.csv", index=False)
df_nbt_en.to_csv("../../../data/LVVI3/lvvi3_nbt.csv", index=False)
df_or_en.to_csv("../../../data/LVVI3/lvvi3_or_en.csv", index=False)
df_nbt_en.to_csv("../../../data/LVVI3/lvvi3_nbt_en.csv", index=False)
print("or_en:", df_or_en.shape, "| nbt_en:", df_nbt_en.shape)

or_en: (165, 114) | nbt_en: (286, 118)


## 5. Write the activity mapping middle product

`data/LVVI3_activity_mapping.csv` — every questionnaire option of every one-shot activity
column, with its canonical `sub_activity`, `activity` and `structural_activity`.

In [15]:
activity_mapping = (
    pd.DataFrame(
        [(section, f"{section}_{stem}", cat, code,
          OPTION_LABELS.get((section, stem, code)),
          SUB_ACTIVITY_CANONICAL.get((section, stem, code)))
         for section in ("B1", "B2")
         for cat, stem in ACTIVITY_COLUMNS
         for code in sorted({c for (s, st, c) in OPTION_LABELS if s == section and st == stem})],
        columns=["section", "survey_column", "activity_cat", "sub_activity_cat",
                 "sub_activity_option", "sub_activity"],
    )
    .assign(activity=lambda d: d["sub_activity"].map(sub_to_activity),
            structural_activity=lambda d: d["activity"].map(STRUCTURAL_BY_ACTIVITY))
)
assert activity_mapping[["sub_activity", "activity", "structural_activity"]].notna().all().all()
activity_mapping.to_csv("../../../data/LVVI3_activity_mapping.csv", index=False)
print("mapping middle product:", activity_mapping.shape)
activity_mapping.head(10)

mapping middle product: (172, 8)


,section,survey_column,activity_cat,sub_activity_cat,sub_activity_option,sub_activity,activity,structural_activity
0,B1,B1_kavely,1,1,overnight hiking,Hiking,Walking,Leisure
1,B1,B1_kavely,1,2,hiking at least 10 km,Hiking,Walking,Leisure
2,B1,B1_kavely,1,3,Nordic walking,Nordic walking,Walking,Leisure
3,B1,B1_kavely,1,4,other walking for exercise,Walking,Walking,Leisure
4,B1,B1_pyora,2,1,mountain biking mainly on trails,Mountain biking,Cycling,Adventure
5,B1,B1_pyora,2,2,cycling tour of at least 30 km or at least hal...,Cycling,Cycling,Adventure
6,B1,B1_pyora,2,3,other recreational cycling or leisure biking,Cycling,Cycling,Adventure
7,B1,B1_juoksu,3,1,jogging or running on walking paths and outdoo...,Jogging,Running,Leisure
8,B1,B1_juoksu,3,2,trail or cross-country running,Running,Running,Leisure
9,B1,B1_koira,4,1,dog walking in nature,Dog walking,Dog walking,Leisure


## 6. Combine the sections and build the main survey table

In [16]:
df = pd.concat([df_or_en.assign(source="or"), df_nbt_en.assign(source="nbt")], ignore_index=True)
print(f"Outdoor recreation rows: {(df.source == 'or').sum()}")
print(f"Nature-based tourism rows: {(df.source == 'nbt').sum()}")
print(f"Combined: {df.shape}")

# Demographic variables
gender_map = {1: "Male", 2: "Female", 3: "Other"}
df["gender"] = df["A1_gender"].map(gender_map)
age_map = {1: "<16", 2: "16-24", 3: "25-34", 4: "35-44",
           5: "45-54", 6: "55-64", 7: "65-74", 8: ">75"}
df["age"] = df["A2_age"].map(age_map)

# sub_activity is already canonical; attach activity / structural_activity
assert df["sub_activity"].notna().all()
assert set(df["sub_activity"]) <= set(sub_to_activity)
df["activity"] = df["sub_activity"].map(sub_to_activity)
df["structural_activity"] = df["activity"].map(STRUCTURAL_BY_ACTIVITY)

# Area-type flags, harmonized with the social media layers. The survey records a
# single declared destination (mutually exclusive) and a single GIS area class, so
# the flags are mutually exclusive, unlike the overlapping spatial-intersection
# flags of YouTube/Flickr:
#   national park     area_destination = 3
#   wilderness area   area_destination = 4
#   nature reserve    area_destination = 5, 6, 7 (state / municipal / other)
#   commercial forest area_classification = 23 (state multiple use; classes 21/22
#                     are state recreation / state protected areas)
df["national_park"] = df["area_destination"].eq(3).astype(int)
df["wilderness_area"] = df["area_destination"].eq(4).astype(int)
df["nature_reserve"] = df["area_destination"].isin([5, 6, 7]).astype(int)
df["commercial_forest"] = df["area_classification"].eq(23).astype(int)

df.to_csv("../../../data/LVVI3/lvvi3_survey_en.csv", index=False)
print("written:", df.shape)

Outdoor recreation rows: 165
Nature-based tourism rows: 286
Combined: (451, 124)
written: (451, 132)


## 7. Validation

- every respondent's filled activity column is the one indicated by `B1_1` / `B2_1`
  (after removing kohdenro 10202 and the code-23 rows);
- number of activities per respondent (one-shot columns filled);
- the decoded activity distribution.

In [17]:
# 7a. consistency between the group code and the filled one-shot column
b1_check = df_lvvi3[df_lvvi3["B1_1"].notnull()].copy()
b1_check["expected_stem"] = b1_check["B1_1"].astype(int).map(
    {cat: stem for cat, stem in ACTIVITY_COLUMNS})
b1_check["filled_stems"] = b1_check.apply(
    lambda r: [stem for _, stem in ACTIVITY_COLUMNS
               if pd.notna(r.get(f"B1_{stem}"))], axis=1)
mismatch = b1_check[b1_check.apply(
    lambda r: r["expected_stem"] not in r["filled_stems"], axis=1)]
print("B1 rows where the filled column does not match B1_1:", len(mismatch))

b2_check = df_lvvi3[df_lvvi3["B2_1"].notnull()].copy()
b2_check["expected_stem"] = b2_check["B2_1"].astype(int).map(
    {cat: stem for cat, stem in ACTIVITY_COLUMNS})
b2_check["filled_stems"] = b2_check.apply(
    lambda r: [stem for _, stem in ACTIVITY_COLUMNS
               if pd.notna(r.get(f"B2_{stem}"))], axis=1)
mismatch2 = b2_check[b2_check.apply(
    lambda r: r["expected_stem"] not in r["filled_stems"], axis=1)]
print("B2 rows where the filled column does not match B2_1:", len(mismatch2))

B1 rows where the filled column does not match B1_1: 0
B2 rows where the filled column does not match B2_1: 0


In [18]:
# 7b. activities per respondent across both sections
acts_per_respondent = pd.concat([
    extract_activities(df_lvvi3[df_lvvi3["B1_1"].notna()], "B1"),
    extract_activities(df_lvvi3[df_lvvi3["B2_1"].notna()], "B2"),
]).groupby("kohdenro").size()
print("activities per respondent:")
print(acts_per_respondent.value_counts().sort_index())

activities per respondent:
1    484
Name: count, dtype: int64


In [19]:
# 7c. decoded activity distribution of the final table
df["activity"].value_counts()

activity
Walking                                                        219
Skiing                                                          40
Dog walking                                                     25
Fishing                                                         23
Cycling                                                         17
Boating                                                         14
Hunting                                                         12
Running                                                         12
Nature observation                                              11
Picnicking                                                      11
Outdoor activities with children                                 8
Nature product collection                                        7
Snowmobiling                                                     6
Orienteering                                                     6
Nature photography                                   